<a href="https://colab.research.google.com/github/Shrutikraj/Deepfake_voice_video_Detection/blob/main/mega_project_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive  #As data is large i am using drive so to not download to its full but should able to use it for that i mount the google drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import glob #Locating the ZIP file path in Drive so to inspect data without extracting or downloading

files = glob.glob("/content/drive/**/*FakeAVCeleb*.zip", recursive=True)
files

In [ ]:
import os # Checking the dataset size

zip_path = "/content/drive/MyDrive/FakeAVCeleb/FakeAVCeleb_v1.2.zip"

size_gb = os.path.getsize(zip_path) / (1024**3)
print("ZIP Size (GB):", round(size_gb, 2))


In [ ]:

import zipfile #Previewing the file and seeing what is inside it

with zipfile.ZipFile(zip_path, 'r') as z:
    names = z.namelist()

print("Total files inside ZIP:", len(names))
names[:30]   # show first 30 files


In [ ]:
import zipfile # Here we are extracting only videos with audio to check
import os

sample_dir = "/content/sample_data_fake_both"
os.makedirs(sample_dir, exist_ok=True)

count = 0

with zipfile.ZipFile(zip_path, 'r') as z:
    for name in names:
        if "FakeVideo-FakeAudio" in name and name.endswith(".mp4"):
            z.extract(name, sample_dir)
            count += 1

        if count >= 5:
            break

print("Extracted", count, "sample files")



In [ ]:
from IPython.display import HTML
from base64 import b64encode
import glob

videos = glob.glob(sample_dir + "/**/*.mp4", recursive=True)

video_path = videos[0]
mp4 = open(video_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=400 controls>
  <source src="{data_url}" type="video/mp4">
</video>
""")


In [ ]:
fake_more_dir = "/content/fake_both_20"  #Extracting 20 samples of fake sampled data
os.makedirs(fake_more_dir, exist_ok=True)

count = 0

with zipfile.ZipFile(zip_path, 'r') as z:
    for name in names:
        if "FakeVideo-FakeAudio" in name and name.endswith(".mp4"):
            z.extract(name, fake_more_dir)
            count += 1

        if count >= 20:
            break

print("Extracted", count, "fake samples")


In [ ]:
real_more_dir = "/content/real_both_20" #Extracting 20 real sampled data
os.makedirs(real_more_dir, exist_ok=True)

count = 0

with zipfile.ZipFile(zip_path, 'r') as z:
    for name in names:
        if "RealVideo-RealAudio" in name and name.endswith(".mp4"):
            z.extract(name, real_more_dir)
            count += 1

        if count >= 20:
            break

print("Extracted", count, "real samples")


In [ ]:
import pandas as pd  #Creating a dataset table with path and label

records = []

for n in names:
    if n.endswith(".mp4"):

        label = None

        if "FakeVideo-FakeAudio" in n:
            label = "fake_video_fake_audio"
        elif "RealVideo-RealAudio" in n:
            label = "real_video_real_audio"
        elif "FakeVideo-RealAudio" in n:
            label = "fake_video_real_audio"
        elif "RealVideo-FakeAudio" in n:
            label = "real_video_fake_audio"

        # skip files that don't belong to these 4 categories
        if label is None:
            continue

        records.append((n, label))

df = pd.DataFrame(records, columns=["path", "label"])
df.head()


In [ ]:
print("Total samples:", len(df)) #Summery of dataset
df["label"].value_counts()


In [ ]:
import matplotlib.pyplot as plt #Visualizing dataset Class Distribution

counts = df["label"].value_counts()

plt.figure()
counts.plot(kind="bar")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.title("Dataset Class Distribution")
plt.show()


In [ ]:
def parse_meta(path):
    parts = path.split("/")

    category = parts[1]
    region = parts[2]
    gender = parts[3]
    person_id = parts[4]

    return region, gender, person_id

df["region"], df["gender"], df["person_id"] = zip(*df["path"].map(parse_meta))

df.head()


In [ ]:
df["region"].value_counts()


In [ ]:
plt.figure()
df["region"].value_counts().plot(kind="bar")
plt.xlabel("Region")
plt.ylabel("Samples")
plt.title("Samples per Region")
plt.show()


In [ ]:
df["gender"].value_counts()



As we want to predict the Fake voice or message in real video but for that perpose the dataset is highly imbalance as sample for real video with real audio or real video with fake audio are comparatively less than data with fake audio and fake video ,etc

We will work with audio data First
and we will use binary way like real and fake voice extraction as we have voice in both real and fake videos and try to build the model for audio prediction first

In [ ]:
audio_records = []

for path, label in zip(df["path"], df["label"]):

    if "real_audio" in label:
        audio_label = "real_audio"

    elif "fake_audio" in label:
        audio_label = "fake_audio"

    else:
        continue

    audio_records.append((path, label, audio_label))

audio_df = pd.DataFrame(audio_records,
                        columns=["video_path", "video_label", "audio_label"])

audio_df.head(), audio_df["audio_label"].value_counts()


In [ ]:
plt.figure()
audio_df["audio_label"].value_counts().plot(kind="bar")
plt.xlabel("Audio Class")
plt.ylabel("Samples")
plt.title("Real vs Fake Audio Distribution")
plt.show()


In [ ]:
import os, json #Load / Create Progress Tracker

progress_file = "processed_index.json"

if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        processed_index = set(json.load(f))
else:
    processed_index = set()

print("Already processed:", len(processed_index))


In [ ]:

BATCH_SIZE = 1000


In [ ]:
import zipfile #Run Batch Extraction
import librosa
import numpy as np

mfcc_batch = []
spec_batch = []
label_batch = []

start_count = len(processed_index)
target_count = start_count + BATCH_SIZE

with zipfile.ZipFile(zip_path, 'r') as z:

    for i, row in audio_df.iterrows():

        if i in processed_index:
            continue   # already done earlier

        if len(processed_index) >= target_count:
            break      # stop batch here

        video_path = row["video_path"]
        label = 0 if row["audio_label"] == "real_audio" else 1

        try:
            # read from zip -> temp file
            with z.open(video_path) as f:
                tmp = "/tmp/temp_video.mp4"
                with open(tmp, "wb") as t:
                    t.write(f.read())

            # load only first 5 seconds audio
            y, sr = librosa.load(tmp, sr=16000, duration=5)

            # ---- MFCC ----
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
            mfcc_delta = librosa.feature.delta(mfcc)
            mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

            mfcc_full = np.concatenate([mfcc, mfcc_delta, mfcc_delta2], axis=0)

            mfcc_vec = np.concatenate([
                np.mean(mfcc_full, axis=1),
                np.std(mfcc_full, axis=1)
            ])

            # ---- Spectrogram ----
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
            log_mel = librosa.power_to_db(mel)

            # pad / crop to 128x128
            if log_mel.shape[1] < 128:
                pad = 128 - log_mel.shape[1]
                log_mel = np.pad(log_mel, ((0,0),(0,pad)))
            else:
                log_mel = log_mel[:, :128]

            mfcc_batch.append(mfcc_vec)
            spec_batch.append(log_mel)
            label_batch.append(label)

            processed_index.add(i)

        except Exception as e:
            print("Skipped:", video_path, "|", e)

print("Batch finished — processed:", len(label_batch))


In [ ]:
with open("audio_features_mfcc.npy", "ab") as f:
    np.save(f, np.array(mfcc_batch))

with open("audio_features_spec.npy", "ab") as f:
    np.save(f, np.array(spec_batch))

with open("audio_labels.npy", "ab") as f:
    np.save(f, np.array(label_batch))

print("Saved batch features")


In [ ]:
with open(progress_file, "w") as f:
    json.dump(list(processed_index), f)

print("Progress saved — total processed:", len(processed_index))


In [ ]:
import numpy as np

def load_npy_stream_safe(path):
    arrays = []
    with open(path, "rb") as f:
        while True:
            try:
                arrays.append(np.load(f, allow_pickle=True))
            except Exception:
                break
    return np.concatenate(arrays)

X_mfcc = load_npy_stream_safe("audio_features_mfcc.npy")
X_spec = load_npy_stream_safe("audio_features_spec.npy")
y = load_npy_stream_safe("audio_labels.npy")

X_mfcc.shape, X_spec.shape, y.shape


In [ ]:
import numpy as np
unique, counts = np.unique(y, return_counts=True)
dict(zip(unique, counts))


all the extracted sample are of fake audio maybe the real and fake samples are in sequence but want to have both now baseline model so extracting real audio features directly rather than extracting everything sequential

In [ ]:
import numpy as np # checking no unextracted real audio samples

unprocessed_real = audio_df[
    (audio_df["audio_label"]=="real_audio") &
    (~audio_df.index.isin(processed_index))
]

len(unprocessed_real)


In [ ]:
TARGET_REAL = 1000 # extracting real audio samples 1000

mfcc_batch = []
spec_batch = []
label_batch = []

count = 0

import zipfile, librosa

with zipfile.ZipFile(zip_path, 'r') as z:

    for i, row in unprocessed_real.iterrows():

        if count >= TARGET_REAL:
            break

        video_path = row["video_path"]
        label = 0   # real_audio

        try:
            with z.open(video_path) as f:
                tmp = "/tmp/temp_video.mp4"
                with open(tmp, "wb") as t:
                    t.write(f.read())

            y, sr = librosa.load(tmp, sr=16000, duration=5)

            # ---- MFCC ----
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
            mfcc_delta = librosa.feature.delta(mfcc)
            mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
            mfcc_full = np.concatenate([mfcc, mfcc_delta, mfcc_delta2], axis=0)

            mfcc_vec = np.concatenate([
                np.mean(mfcc_full, axis=1),
                np.std(mfcc_full, axis=1)
            ])

            # ---- Spectrogram ----
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
            log_mel = librosa.power_to_db(mel)

            if log_mel.shape[1] < 128:
                pad = 128 - log_mel.shape[1]
                log_mel = np.pad(log_mel, ((0,0),(0,pad)))
            else:
                log_mel = log_mel[:, :128]

            mfcc_batch.append(mfcc_vec)
            spec_batch.append(log_mel)
            label_batch.append(label)

            processed_index.add(i)
            count += 1

        except Exception as e:
            print("Skipped:", video_path, "|", e)

print("Added real-audio samples:", count)


In [ ]:
with open("audio_features_mfcc.npy", "ab") as f: #Append this batch to the feature files
    np.save(f, np.array(mfcc_batch))

with open("audio_features_spec.npy", "ab") as f:
    np.save(f, np.array(spec_batch))

with open("audio_labels.npy", "ab") as f:
    np.save(f, np.array(label_batch))

import json
with open("processed_index.json", "w") as f:
    json.dump(list(processed_index), f)

print("Saved — dataset now more balanced")


In [ ]:
X_mfcc = load_npy_stream_safe("audio_features_mfcc.npy") #Reload features & check class balance
X_spec = load_npy_stream_safe("audio_features_spec.npy")
y = load_npy_stream_safe("audio_labels.npy")

np.unique(y, return_counts=True)


In [ ]:
from sklearn.model_selection import train_test_split # Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X_mfcc, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
from sklearn.preprocessing import StandardScaler #Standardizing mfccs features

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
from sklearn.linear_model import LogisticRegression #Train Baseline Classifier

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_scaled, y_train)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report # Model evaluation

y_pred = clf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(
    y_test, y_pred,
    target_names=["real_audio", "fake_audio"]
))


In [ ]:
import joblib #Save Model + Scaler

joblib.dump(clf, "audio_mfcc_classifier_baseline.pkl")
joblib.dump(scaler, "audio_mfcc_scaler.pkl")

print("Model & scaler saved")


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["real_audio","fake_audio"],
            yticklabels=["real_audio","fake_audio"])

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
video_records = []

for path, label in zip(df["path"], df["label"]):

    if "fake_video" in label:
        video_label = "fake_video"

    elif "real_video" in label:
        video_label = "real_video"

    else:
        continue

    video_records.append((path, video_label))


video_df = pd.DataFrame(video_records, columns=["video_path", "video_label"])

video_df["video_label"].value_counts()


In [ ]:
!pip install opencv-python


In [ ]:
real_videos = video_df[video_df["video_label"]=="real_video"]

fake_videos = video_df[video_df["video_label"]=="fake_video"].sample(
    n=len(real_videos), random_state=42
)

balanced_df = pd.concat([real_videos, fake_videos]).reset_index(drop=True)

balanced_df["video_label"].value_counts()


In [ ]:
import cv2
import zipfile
import os

frame_output_dir = "/content/video_frames"
os.makedirs(frame_output_dir, exist_ok=True)

frames_per_video = 3
size = (128, 128)

with zipfile.ZipFile(zip_path, 'r') as z:

    for i, row in balanced_df.iterrows():

        path_in_zip = row["video_path"]
        label = row["video_label"]

        label_dir = os.path.join(frame_output_dir, label)
        os.makedirs(label_dir, exist_ok=True)

        try:
            with z.open(path_in_zip) as f:
                tmp = "/tmp/temp_video.mp4"
                with open(tmp, "wb") as t:
                    t.write(f.read())

            cap = cv2.VideoCapture(tmp)
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            if frame_count == 0:
                continue

            step = max(frame_count // (frames_per_video+1), 1)

            fno = 0
            saved = 0

            while saved < frames_per_video and cap.isOpened():
                cap.set(cv2.CAP_PROP_POS_FRAMES, fno)
                ret, frame = cap.read()

                if not ret:
                    break

                frame = cv2.resize(frame, size)
                fname = f"{i}_{saved}.jpg"
                cv2.imwrite(os.path.join(label_dir, fname), frame)

                saved += 1
                fno += step

            cap.release()

        except Exception as e:
            print("Skipped:", path_in_zip, "|", e)

print("Balanced frame extraction complete")


In [ ]:
import glob
import numpy as np

X = []
y = []

for label, idx in {"real_video":0, "fake_video":1}.items():

    files = glob.glob(f"/content/video_frames/{label}/*.jpg")

    for f in files:
        img = cv2.imread(f)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (128, 128))
        img = img.astype("float32") / 255.0

        X.append(img)
        y.append(idx)

X = np.array(X)
y = np.array(y)

X.shape, y.shape


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(128,128,3)),
    MaxPooling2D(),

    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(),

    Flatten(),
    Dropout(0.3),
    Dense(128, activation="relu"),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=8,
    batch_size=32
)


In [ ]:
model.save("video_cnn_baseline.h5")
print("Baseline video model saved")


In [ ]:
from tensorflow.keras.models import load_model
video_model = load_model("video_cnn_baseline.h5")
audio_model = joblib.load("audio_mfcc_classifier_baseline.pkl")
scaler = joblib.load("audio_mfcc_scaler.pkl")



In [ ]:
def extract_center_frame(video_path):

    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count == 0:
        raise ValueError("No frames in video")

    middle = frame_count // 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, middle)

    ret, frame = cap.read()
    cap.release()

    frame = cv2.resize(frame, (128,128))
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = frame.astype("float32") / 255.0

    return np.expand_dims(frame, axis=0)


In [ ]:
def predict_video_authenticity(video_path):

    frame = extract_center_frame(video_path)

    prob = video_model.predict(frame)[0][0]

    if prob >= 0.5:
        label = "FAKE VIDEO"
    else:
        label = "REAL VIDEO"

    return label, prob


In [ ]:
def multimodal_fake_detection(video_path):

    audio_label, audio_prob, _ = predict_audio_authenticity(video_path)
    video_label, video_prob = predict_video_authenticity(video_path)

    result = {
        "audio_label": audio_label,
        "audio_prob": audio_prob,
        "video_label": video_label,
        "video_prob": video_prob
    }

    # Final Case Classification

    if audio_label.startswith("REAL") and video_label.startswith("REAL"):
        result["case"] = "REAL VIDEO + REAL AUDIO"

    elif audio_label.startswith("FAKE") and video_label.startswith("REAL"):
        result["case"] = "REAL VIDEO + FAKE AUDIO"

    elif audio_label.startswith("REAL") and video_label.startswith("FAKE"):
        result["case"] = "FAKE VIDEO + REAL AUDIO"

    else:
        result["case"] = "FAKE VIDEO + FAKE AUDIO"

    return result


In [ ]:
out = multimodal_fake_detection(video_path)

for k,v in out.items():
    print(k,":",v)


In [ ]:
!pip install mediapipe
